In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.preprocessing import MinMaxScaler

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/'
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2','IPS']

In [ ]:
snmct_traj = pd.read_csv(f'{projdir}/csv/traj_score/rna_mc_traj_score.csv', sep='\t', index_col=0)
snmct_traj.shape

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc'
snm3c_traj = pd.read_csv(f'{indir}/csv/traj_score/traj_score.csv', sep='\t', index_col=0).dropna()
snmct_traj.rename(columns={'rna_cluster' : 'cluster'}, inplace=True)
snm3c_traj.shape

In [ ]:
snmct_traj.head()

In [ ]:
scaler = MinMaxScaler()
scaled_traj = np.ravel(scaler.fit_transform(snmct_traj['traj_score'].values.reshape(-1, 1)))
snmct_traj['scaled_traj'] = scaled_traj

In [ ]:
snmct_traj['cluster'].unique()

In [ ]:
snm3c_traj.head()

In [ ]:
m3c_clusts = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
m3c_clusts.shape

In [ ]:
snm3c_traj['cluster'] = m3c_clusts.reindex(snm3c_traj.index)['cluster']
snm3c_traj['cluster'].unique()

In [ ]:
snmct_traj['assay'] = 'snmCT'
snm3c_traj['assay'] = 'snm3C'

In [ ]:
select = ['scaled_traj', 'cluster', 'assay'] 
merged = pd.concat([snmct_traj[select], snm3c_traj[select]])
merged.head()

In [ ]:
blacklist = ['Skin', 'Inflamed']
mask = [x not in blacklist for x in merged['cluster']]
final_merged = merged[mask]
final_merged.shape

In [ ]:
to_plot = final_merged.melt(id_vars=['cluster', 'assay'], value_vars=['scaled_traj'])
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 6))

x_lab = 'cluster'
y_lab = 'value'
hue = 'assay'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue,
                 order=tmp_order)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('Trajectory score', fontsize=16)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelsize=14, labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 6))

x_lab = 'cluster'
y_lab = 'value'
hue = 'assay'

ax = sns.lineplot(data=to_plot, x=x_lab, y=y_lab, hue=hue)
ax.set_xlabel('Cluster', fontsize=16)
ax.set_ylabel('Trajectory score', fontsize=16)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelsize=14, labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
to_plot

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
to_plot.to_csv(f'{outdir}/mct_m3c_traj_score.csv', sep='\t')

In [ ]:
!date